# DuckGPT — 04. Training dynamics

We train the tiny GPT from `core/05_model.py` on Shakespeare-char for a
few hundred iterations and plot:

- training loss
- validation loss
- learning rate schedule (cosine + warm-up)

This is the same loop as `core/07_train.py`, but inlined so we can plot.

In [ ]:
import os, sys, pickle
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.append('../core')
from importlib import import_module
m_model = import_module('05_model')
m_optim = import_module('06_optim')
GPT, GPTConfig = m_model.GPT, m_model.GPTConfig
Adam = m_optim.Adam
cosine_warmup_lr = m_optim.cosine_warmup_lr

DATA_DIR = Path('..') / 'data' / 'shakespeare_char'
with open(DATA_DIR / 'meta.pkl', 'rb') as f:
    meta = pickle.load(f)
vocab_size = meta['vocab_size']
train_data = np.fromfile(DATA_DIR / 'train.bin', dtype=np.uint16)
val_data   = np.fromfile(DATA_DIR / 'val.bin',   dtype=np.uint16)
vocab_size, len(train_data), len(val_data)

In [ ]:
batch = 32; block = 64
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = np.random.randint(0, len(data) - block - 1, size=batch)
    x = np.stack([data[i:i+block] for i in ix]).astype(np.int64)
    y = np.stack([data[i+1:i+1+block] for i in ix]).astype(np.int64)
    return torch.from_numpy(x), torch.from_numpy(y)

cfg = GPTConfig(vocab_size=vocab_size, block_size=block,
                n_layer=3, n_head=4, n_embd=96)
torch.manual_seed(0)
model = GPT(cfg)
opt = Adam(model.parameters(), lr=3e-3, weight_decay=0.1)
max_iters = 300; warmup = 30; lr_max = 3e-3; lr_min = 1e-4
train_curve, val_curve, lr_curve = [], [], []
for it in range(max_iters):
    opt.lr = cosine_warmup_lr(it, warmup, max_iters, lr_max, lr_min)
    lr_curve.append(opt.lr)
    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    train_curve.append(loss.item())
    opt.zero_grad(); loss.backward(); opt.step()
    if it % 20 == 0:
        with torch.no_grad():
            xb, yb = get_batch('val')
            _, vl = model(xb, yb)
            val_curve.append((it, vl.item()))
len(train_curve), len(val_curve)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(train_curve, label='train', alpha=0.7)
vi, vv = zip(*val_curve)
ax[0].plot(vi, vv, 'o-', label='val')
ax[0].set_xlabel('iter'); ax[0].set_ylabel('loss'); ax[0].set_title('losses')
ax[0].legend()
ax[1].plot(lr_curve)
ax[1].set_xlabel('iter'); ax[1].set_ylabel('lr'); ax[1].set_title('warmup + cosine schedule')
plt.tight_layout(); plt.show()

## Things to try

- Disable warm-up — what happens to the early-iteration loss?
- Make `lr_max` much larger (e.g. 0.1) — at what point does training diverge?
- Make the model wider (`n_embd=192`) — how much does val loss improve?